[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_04/4_ai.ipynb)

# Day 4: AI and ML (easy)

**How to work:** Answer concept questions out loud in 60 to 90 seconds before opening the solution: definition, why it matters, one example, one trade-off.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Why not just fit a line to 0/1? | easy | 4 |
| Q2 | Squash and score | easy | 5 |
| Q3 | Read the odds on the Titanic | easy | 7 |
| Q4 | Who is about to leave? | easy | 6 |
| Q5 | The coefficient with the wrong sign (review) | easy | 3 |

**Handbook:** [AI and ML](https://mahsa-agz.github.io/ds-handbook/#ai), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `titanic`: 891 passengers, one row = one passenger. `survived` (0/1), `pclass`, `sex`, `age` (has NaN), `sibsp`, `parch`, `fare`, `embarked`, plus derived columns (`alive`, `who`, ...).
- `telco`: IBM Telco churn, 7,043 customers, one row = one customer. `Churn` (Yes/No), `tenure` (months), `Contract`, `MonthlyCharges`, `TotalCharges` (text with blanks), 9 service columns, demographics.

The setup cell loads every table into a pandas DataFrame (from your Drive folder, or downloads it). Everything runs on the free Colab CPU in seconds. Numbers in the solutions come from scikit-learn 1.8; other versions can differ in the last digit.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import io, zipfile, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 120)

SEABORN = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

def load_telco():
    """IBM Telco churn, 7,043 customers. Target: Churn (Yes/No)."""
    return pd.read_csv(data_path("telco_churn.csv", TELCO_URL))

def load_seaborn(name):
    """titanic, penguins, diamonds, mpg, tips ... from the seaborn-data repo."""
    return pd.read_csv(data_path(name + ".csv", SEABORN + name + ".csv"))

def load_bank():
    """UCI Bank Marketing (bank-additional-full), 41,188 calls. Target: y (yes/no). The source is a zip in a zip."""
    path = data_path("bank_marketing.csv")
    if not os.path.exists(path):
        print("downloading bank_marketing (zip)")
        outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(BANK_URL).read()))
        inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
        with open(path, "wb") as f:
            f.write(inner.read("bank-additional/bank-additional-full.csv"))
    return pd.read_csv(path, sep=";")

def check(fn, cases, tol=1e-6):
    """Runs fn on each (args, expected) case; numbers and arrays are compared with a tolerance."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*args)
            good = got is not None and np.shape(got) == np.shape(want) and bool(
                np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), atol=tol))
        except Exception as e:
            got, good = "error: " + repr(e)[:120], False
        ok += good
        shown = " ".join(repr(got).split())
        shown = shown if len(shown) < 100 else shown[:97] + "..."
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}(...) -> {shown}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

titanic = load_seaborn("titanic")
print("titanic", titanic.shape)
telco = load_telco()
print("telco", telco.shape)

---
### Q1. Why not just fit a line to 0/1?

*easy, about 4 min*

Explain logistic regression in about 90 seconds: why not use linear regression on a 0/1 target, the model formula, the loss it minimises, and how to read a coefficient.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: binary target, 'probability', 'odds'. Pattern: logistic regression = linear model on the log-odds.

</details>

<details><summary><b>Hint 2</b></summary>

1. Linear regression gives values outside [0, 1] and assumes constant variance. 2. Model: `log(p / (1 - p)) = b0 + b x`, so `p = 1 / (1 + exp(-z))`. 3. Loss: log loss (cross-entropy), from maximum likelihood. 4. `exp(b)` = odds ratio for a one-unit increase.

</details>

<details><summary><b>Solution</b></summary>

**Model answer (about 90 s):**

"With a 0/1 target, a straight line predicts values below 0 and above 1, the errors cannot be normal or constant-variance, and outliers in x pull the line and move the decision boundary. Logistic regression keeps a linear score `z = b0 + b x` but maps it through the sigmoid, `p = 1 / (1 + exp(-z))`, so the output is a probability. Equivalently, the **log-odds** `log(p / (1 - p))` are linear in x.

It is fitted by maximum likelihood, which means minimising the log loss `-mean(y log p + (1 - y) log(1 - p))`. That loss is convex, so there is one optimum, found with gradient methods; there is no closed form. The gradient has a neat form: `X^T (p - y)`.

A coefficient b means: one more unit of x multiplies the odds by `exp(b)`, holding the others fixed. For example b = 0.7 gives an odds ratio of about 2. It is not a fixed change in probability: the probability effect is largest near p = 0.5. The decision boundary `z = 0` is linear, so you need feature engineering or interactions for curved boundaries."

**Why:** The three must-say points: sigmoid of a linear score, log loss from maximum likelihood, and odds-ratio interpretation. Bonus: convexity and the `X^T (p - y)` gradient.

**Common mistakes:** Saying the coefficient is the change in probability. Saying logistic regression is fitted with squared error. Calling it a regression model that 'cannot classify' (it is a classifier plus a threshold).

**Learn more:** [ai-logistic-regression](https://mahsa-agz.github.io/ds-handbook/#ai-logistic-regression)

</details>

---
### Q2. Squash and score

*easy, about 5 min*

Write two numpy functions:

- `sigmoid(z)`: works on scalars and arrays and does not overflow for `z = -1000` or `z = 1000`.
- `log_loss(y, p)`: the mean binary cross-entropy. Clip p to `[1e-15, 1 - 1e-15]` so a perfect prediction does not give `log(0)`.

Example: `sigmoid(0) = 0.5`, `sigmoid(log 3) = 0.75`, `log_loss([1, 0], [0.5, 0.5]) = log 2 = 0.693`.

In [ ]:
def sigmoid(z):
    # your code here
    pass

def log_loss(y, p):
    # your code here
    pass

# tests
check(sigmoid, [(0, 0.5), (np.log(3), 0.75), (([-1000.0, 1000.0],), [0.0, 1.0])])
check(log_loss, [(([1, 0], [0.5, 0.5]), np.log(2)),
                 (([1, 1, 0], [0.9, 0.8, 0.3]), 0.22839300363692283),
                 (([1, 0], [1.0, 0.0]), 1e-15)], tol=1e-9)

<details><summary><b>Hint 1</b></summary>

Signal: probabilities from scores, and a loss for probabilities. Pattern: logistic function and cross-entropy, with numerical stability.

</details>

<details><summary><b>Hint 2</b></summary>

1. `1 / (1 + exp(-z))` overflows for very negative z; for z < 0 use `exp(z) / (1 + exp(z))`. 2. Log loss: `-mean(y log p + (1 - y) log(1 - p))` after clipping p.

</details>

<details><summary><b>Solution</b></summary>

```python
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    # exp(-|z|) never overflows; use the right algebraic form for each sign
    e = np.exp(-np.abs(z))
    return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

def log_loss(y, p, eps=1e-15):
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)   # avoid log(0)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
```

**Why:** `sigmoid(log 3) = 1 / (1 + 1/3) = 0.75`: log 3 is the log-odds of 3 to 1. Writing the sigmoid with `exp(-|z|)` keeps every exponent at most 0, so nothing overflows. Log loss punishes confident wrong answers hard: predicting 0.01 for a positive costs `-log(0.01) = 4.6`, while 0.5 costs only 0.69. That is why it rewards calibrated probabilities, not only correct labels.

**Complexity:** `O(n)` time and memory.

**Common mistakes:** Not clipping p, so one perfect wrong prediction gives infinity. Using the natural log in one place and log2 in another. `np.exp(-z)` overflow warnings for large negative z.

**Learn more:** [ai-logistic-regression](https://mahsa-agz.github.io/ds-handbook/#ai-logistic-regression), [ai-classification-metrics](https://mahsa-agz.github.io/ds-handbook/#ai-classification-metrics)

</details>

---
### Q3. Read the odds on the Titanic

*easy, about 7 min*

Use `titanic`, keeping only rows with a known `age`. Fit a logistic regression of `survived` on `sex`, `pclass` (as a category) and `age`, without regularization (statsmodels `smf.logit` is ideal).

1. Print the number of rows used and, for each coefficient, the value and the odds ratio `exp(coef)`.
2. Print the predicted survival probability of a 30-year-old woman and a 30-year-old man, both in 3rd class.

Interpret the `male` and `age` odds ratios in one sentence each.

In [ ]:
import statsmodels.formula.api as smf

# your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'interpret the coefficients'. Pattern: logistic regression with odds ratios, `exp(b)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. `smf.logit('survived ~ C(sex) + C(pclass) + age', data=d).fit(disp=0)`. 2. Loop over `fit.params`. 3. `fit.predict(new_df)` gives probabilities.

</details>

<details><summary><b>Solution</b></summary>

```python
import statsmodels.formula.api as smf

d = titanic.dropna(subset=["age"])
fit = smf.logit("survived ~ C(sex) + C(pclass) + age", data=d).fit(disp=0)
print("rows used:", len(d))   # out: rows used: 714
for name, b in fit.params.items():
    print(f"{name:20s} coef {b:+.3f}  odds ratio {np.exp(b):.3f}")
# out: Intercept            coef +3.777  odds ratio 43.685
# out: C(sex)[T.male]       coef -2.523  odds ratio 0.080
# out: C(pclass)[T.2]       coef -1.310  odds ratio 0.270
# out: C(pclass)[T.3]       coef -2.581  odds ratio 0.076
# out: age                  coef -0.037  odds ratio 0.964

# predicted survival probability for two 30-year-olds in 3rd class
new = pd.DataFrame({"sex": ["female", "male"], "pclass": [3, 3], "age": [30, 30]})
print(fit.predict(new).round(3).tolist())   # out: [0.522, 0.08]
```

**Why:** Holding class and age fixed, being male multiplies the odds of survival by 0.080, about 12 times lower odds. Each extra year of age multiplies the odds by 0.964, about 3.6% lower odds per year, so 10 years gives `0.964^10 = 0.69`. The two passengers differ only in sex, yet their probabilities are 0.522 versus 0.08: the odds ratio of 0.080 does not mean 'probability times 0.08', because odds = p / (1 - p) (`0.522 / 0.478 = 1.09` and `0.080 / 0.920 = 0.087`, ratio 0.08).

**Complexity:** Newton steps on 714 rows and 5 parameters: milliseconds.

**Common mistakes:** Treating `pclass` as a number (forces equal steps from 1st to 2nd and 2nd to 3rd class). Reading odds ratios as probability ratios. Dropping missing ages without saying it changes the sample (177 passengers removed).

**Learn more:** [ai-logistic-regression](https://mahsa-agz.github.io/ds-handbook/#ai-logistic-regression), [stats-regression](https://mahsa-agz.github.io/ds-handbook/#stats-regression)

</details>

---
### Q4. Who is about to leave?

*easy, about 6 min*

Use `telco`. Build four features: tenure in **years**, monthly charges in **units of 10 dollars**, and two 0/1 columns for `One year` and `Two year` contracts (month-to-month is the reference). Fit a logistic regression with practically no regularization (`C=1e6`) on all rows.

1. Print the odds ratio of each feature.
2. Print the churn probability of (a) a month-to-month customer in month 1 paying 90 dollars and (b) a two-year-contract customer with 5 years of tenure paying 90 dollars.

Why did we rescale tenure and charges before reading the odds ratios?

In [ ]:
from sklearn.linear_model import LogisticRegression

# telco is loaded. Your code here

<details><summary><b>Hint 1</b></summary>

Signal: a business question about drivers plus a probability for a specific customer. Pattern: logistic regression, odds ratios per meaningful unit.

</details>

<details><summary><b>Hint 2</b></summary>

1. Build the four columns in a DataFrame. 2. `LogisticRegression(C=1e6, max_iter=1000).fit(X, y)`. 3. `np.exp(lr.coef_[0])`. 4. `predict_proba(new)[:, 1]`.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.linear_model import LogisticRegression

X = pd.DataFrame({
    "tenure_years": telco["tenure"] / 12,
    "monthly_per_10usd": telco["MonthlyCharges"] / 10,
    "one_year": (telco["Contract"] == "One year").astype(int),
    "two_year": (telco["Contract"] == "Two year").astype(int),
})
y = (telco["Churn"] == "Yes").astype(int)
lr = LogisticRegression(C=1e6, max_iter=1000).fit(X, y)   # huge C = practically no regularization
for name, b in zip(X.columns, lr.coef_[0]):
    print(f"{name:18s} odds ratio {np.exp(b):.3f}")
# out: tenure_years       odds ratio 0.651
# out: monthly_per_10usd  odds ratio 1.331
# out: one_year           odds ratio 0.346
# out: two_year           odds ratio 0.133

two = pd.DataFrame({"tenure_years": [1 / 12, 5.0], "monthly_per_10usd": [9.0, 9.0],
                    "one_year": [0, 0], "two_year": [0, 1]})
print(lr.predict_proba(two)[:, 1].round(3).tolist())   # out: [0.719, 0.039]
```

**Why:** Each extra year of tenure cuts the odds of churn by about 35% (0.651), every extra 10 dollars a month raises them by about 33% (1.331), and a two-year contract has 0.133 times the odds of a month-to-month one at the same tenure and price. The two example customers sit at 0.719 versus 0.039. Rescaling chooses the unit the odds ratio refers to: per month the tenure odds ratio would be `0.651^(1/12) = 0.965`, which looks tiny but is the same effect. Pick units a PM thinks in. Note sklearn regularizes by default (`C=1.0`); for interpretation either turn it down, as here, or standardise the features so the penalty treats them equally.

**Complexity:** Four features, 7,043 rows: milliseconds.

**Common mistakes:** Reading coefficients from a default sklearn model as if they were unpenalised. Comparing raw coefficients of features on different scales to rank 'importance'. Reading the contract effect as causal (customers choose their contract).

**Learn more:** [ai-logistic-regression](https://mahsa-agz.github.io/ds-handbook/#ai-logistic-regression)

</details>

---
### Q5. The coefficient with the wrong sign (review)

*easy, about 3 min*

A linear regression predicts house price from `square_meters` and `number_of_rooms`. The coefficient on rooms is **negative**, although houses with more rooms sell for more. A stakeholder says the model is broken. Explain what happens and what you would do.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: two features that move together and a counter-intuitive sign. Pattern: multicollinearity and 'holding the other fixed' interpretation.

</details>

<details><summary><b>Hint 2</b></summary>

1. A coefficient is the effect with the other features held constant. 2. At fixed size, more rooms means smaller rooms. 3. Correlated features inflate coefficient variance (check VIF).

</details>

<details><summary><b>Solution</b></summary>

**Model answer:** "The model may be right. A coefficient in a multiple regression is the effect of one feature **holding the others fixed**. At the same square meters, an extra room means the space is cut into smaller rooms, which buyers may value less, so a negative sign is plausible. The univariate correlation of rooms with price is positive because rooms is a proxy for size.

Also, size and rooms are strongly correlated (multicollinearity), so the two coefficients are estimated with large variance and can swing or flip between samples, even though predictions stay good. I would check the variance inflation factor (VIF above about 5 to 10 is a warning), bootstrap the coefficients to see if the sign is stable, and if the goal is interpretation, use one size feature or a ratio like `square_meters / rooms`, or ridge regression to stabilise them."

**Why:** This tests whether you can read coefficients 'all else equal' and separate prediction quality from coefficient stability.

**Common mistakes:** Dropping a feature only because its sign looks wrong. Saying multicollinearity biases the coefficients (it inflates their variance).

**Learn more:** [ai-linear-regression](https://mahsa-agz.github.io/ds-handbook/#ai-linear-regression), [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Done for today.** Finish with the [theory flashcards for day 4](https://mahsa-agz.github.io/ds-handbook/#day-4).